<a href="https://colab.research.google.com/github/sicelomathambo-ship-it/fittrack-ai/blob/main/FitAI_%E2%80%93_Calories_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FitAI – Calories Prediction Model

## Project Overview
We developed a machine learning model to predict calories burned during a workout session using user attributes such as age, gender, weight, heart rate, and workout type.

## Model Approach
We used a Random Forest regression model trained on a Kaggle dataset of gym member exercise data.

## Results
- Mean Absolute Error (MAE): ~35 calories  
- R² Score: ~0.97  

## Business Value
This model can be integrated into a fitness application to help users estimate workout outcomes, optimize training plans, and make informed fitness decisions.

## Next Steps
The trained model will be exported and integrated into a web-based user interface as part of Phase B, with an LLM layer added for explainability and governance in Phase C.

## Data Loading

### Uploading a Dataset

To upload your dataset, run the following cell. It will provide a button to browse and select files from your local machine. Once uploaded, the file will be available in the Colab environment.

In [ ]:
import pandas as pd
from google.colab import files
import io
import zipfile
import os

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
  if fn.endswith('.zip'):
    # It's a zip file, so extract it first
    with zipfile.ZipFile(io.BytesIO(uploaded[fn]), 'r') as zip_ref:
      zip_ref.extractall('.') # Extract to the current directory
    print(f'Successfully unzipped "{fn}"')

    # Find the CSV file within the unzipped contents
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
      csv_filename = csv_files[0] # Assuming there's one CSV file in the zip
      df = pd.read_csv(csv_filename)
      print(f'Successfully loaded "{csv_filename}" into a DataFrame. Here are the first 5 rows:')
      display(df.head())
    else:
      print("No CSV file found within the uploaded zip archive.")
  elif fn.endswith('.csv'):
    # Assuming the uploaded file is a CSV, you can load it into a pandas DataFrame
    df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
    print(f'Successfully loaded "{fn}" into a DataFrame. Here are the first 5 rows:')
    display(df.head())
  else:
    print(f"Unsupported file type: {fn}. Please upload a .csv or .zip file containing a .csv.")

KeyboardInterrupt: 

## Data Exploration (Minimal)

In [ ]:
print(df.info())
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            973 non-null    int64  
 1   Gender                         973 non-null    object 
 2   Weight (kg)                    973 non-null    float64
 3   Height (m)                     973 non-null    float64
 4   Max_BPM                        973 non-null    int64  
 5   Avg_BPM                        973 non-null    int64  
 6   Resting_BPM                    973 non-null    int64  
 7   Session_Duration (hours)       973 non-null    float64
 8   Calories_Burned                973 non-null    float64
 9   Workout_Type                   973 non-null    object 
 10  Fat_Percentage                 973 non-null    float64
 11  Water_Intake (liters)          973 non-null    float64
 12  Workout_Frequency (days/week)  973 non-null    int

,Age,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
count,973.000000,973.000000,973.00000,973.000000,973.000000,973.000000,973.000000,973.000000,973.000000,973.000000,973.000000,973.000000,973.000000
mean,38.683453,73.854676,1.72258,179.883864,143.766701,62.223022,1.256423,905.422405,24.976773,2.626619,3.321686,1.809866,24.912127
std,12.180928,21.207500,0.12772,11.525686,14.345101,7.327060,0.343033,272.641516,6.259419,0.600172,0.913047,0.739693,6.660879
min,18.000000,40.000000,1.50000,160.000000,120.000000,50.000000,0.500000,303.000000,10.000000,1.500000,2.000000,1.000000,12.320000
25%,28.000000,58.100000,1.62000,170.000000,131.000000,56.000000,1.040000,720.000000,21.300000,2.200000,3.000000,1.000000,20.110000
50%,40.000000,70.000000,1.71000,180.000000,143.000000,62.000000,1.260000,893.000000,26.200000,2.600000,3.000000,2.000000,24.160000
75%,49.000000,86.000000,1.80000,190.000000,156.000000,68.000000,1.460000,1076.000000,29.300000,3.100000,4.000000,2.000000,28.560000
max,59.000000,129.900000,2.00000,199.000000,169.000000,74.000000,2.000000,1783.000000,35.000000,3.700000,5.000000,3.000000,49.840000


## Data Exploration Summary

The dataset contains 973 entries and 15 columns, with no missing values across all features.

Key observations:
- Age ranges from 18 to 59 (avg ~38.7)
- Weight ranges from 40kg to 129.9kg (avg ~73.85kg)
- Height ranges from 1.5m to 2.0m (avg ~1.72m)
- Calories burned ranges from 303 to 1783 (avg ~905)

Additional features such as heart rate metrics, workout frequency, BMI, and experience level show reasonable distributions for fitness data.

Overall, the dataset appears clean and well-structured, making it suitable for model training without extensive preprocessing.

In [ ]:
# Select features (X) and target (y)
X = df.drop("Calories_Burned", axis=1)
y = df["Calories_Burned"]

In [ ]:
X = pd.get_dummies(X, drop_first=True)

## Train/Test Split

## Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(778, 16)
(195, 16)


## Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, y_train)

RandomForestRegressor()

## Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 35.60861538461538
R2: 0.9724302942608379


## Save Model

In [ ]:
import joblib

joblib.dump(model, "calories_model.joblib")

['calories_model.joblib']

In [ ]:
This cell was a placeholder for an additional model training step, but as the model has already been trained, evaluated, and saved in the previous steps, this cell is not needed to avoid redundancy.

RandomForestRegressor()

## Model Summary

We built a Random Forest regression model to predict calories burned using workout and biometric data.

The model achieved strong performance:
- MAE ≈ 35 calories
- R² ≈ 0.97

The trained model was exported for integration into the FitAI application.